# Phase 3 Task 1 - YOLO + ByteTrack perception frontend (Colab T4 execution interface)

**Status of this notebook: NOT YET EXECUTED.** No GPU validation exists until you run it on Colab and keep its outputs.

This notebook is an *execution interface only*. It unpacks the source ZIP built from the local repository, verifies its SHA-256
hashes, installs pinned dependencies, downloads the YOLO weights (Colab only), then calls the same
`scripts/run_perception.py` and `scripts/benchmark_perception.py` that run locally. It contains no perception implementation.

Before running:
1. Upload `pedsafety_task1_bundle.zip` **and** `pedsafety_task1_bundle.manifest.json` to Google Drive (default folder `MyDrive/pedsafety_task1/`).
2. Optionally upload a short road video you have rights to and set `INPUT_VIDEO` below. Without it, the YOLO run uses the synthetic
   rectangle video: GPU plumbing and timing only. Its benchmark is marked `--plumbing-only` (can never support a performance claim) and
   **real-video validation is reported as BLOCKED**.
3. Paste the ZIP SHA-256 from the handoff report into `EXPECTED_ZIP_SHA256`.
4. Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all (approve the Drive prompt).

Where things go: all work happens on local Colab storage under `/content/work/<UTC>_<name>/` (annotated MP4 and JSONL are never written directly to
Drive, and a real input video is copied off Drive first). When the run ends - or **fails at any stage** - the whole folder is copied to a new
timestamped folder under `DRIVE_ROOT/experiments/`, read back and checked. `colab_run_manifest.json` is written on success *and* on failure
(failed stage, error text, environment).

In [ ]:
# ---- 1. PARAMETERS (edit this cell only) ------------------------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/pedsafety_task1"          # folder that holds the bundle ZIP + manifest; experiments are copied below it
BUNDLE_ZIP = "pedsafety_task1_bundle.zip"
BUNDLE_MANIFEST = "pedsafety_task1_bundle.manifest.json"
EXPECTED_ZIP_SHA256 = ""        # paste the ZIP SHA-256 from the handoff report (strongly recommended); empty = only compare with the manifest
INPUT_VIDEO = ""                # Drive path to a real road video, e.g. DRIVE_ROOT + "/videos/road_clip.mp4"; empty = synthetic plumbing video
WEIGHTS_FILE = "yolo11n.pt"     # "yolo11n.pt" or "yolo11s.pt"
WEIGHTS_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/" + WEIGHTS_FILE   # UNVERIFIED: if it fails the manifest records BLOCKED
MIN_WEIGHTS_BYTES = 1_000_000   # basic sanity check on the downloaded file (real weights are several MB; an error page is a few KB)
MAX_FRAMES = 300                # frames processed per pass (None = whole video)
WARMUP = 5                      # dummy inferences excluded from timing
DEVICE = "cuda"                 # 'cuda' fails loudly if no GPU; do not switch to 'cpu' for GPU claims
EXPERIMENT_NAME = "t4_perception"
WORK_ROOT = "/content/work"     # local Colab storage; everything is produced here first

In [ ]:
# ---- 2. BOOTSTRAP: status tracking and a failure-proof manifest writer (nothing here depends on the uploaded source) ---------
import contextlib, datetime, hashlib, json, os, pathlib, platform, shutil, subprocess, sys, traceback, urllib.error, zipfile

NOW = lambda: datetime.datetime.now(datetime.timezone.utc)
STAMP = NOW().strftime("%Y%m%dT%H%M%SZ")
LOCAL_EXP = pathlib.Path(WORK_ROOT) / f"{STAMP}_{EXPERIMENT_NAME}"      # all outputs are produced here
(LOCAL_EXP / "logs").mkdir(parents=True, exist_ok=False)
DRIVE_EXP = None                                                       # set once Drive is mounted
STATUS = {"started_utc": NOW().isoformat(), "steps": {}, "blocked_validations": {}}
ERRORS = []
CTX = {}                                                               # facts gathered along the way (zip hash, weights, video, env ...)
DRIVE_COPY = {"attempted": False, "ok": None, "files": 0, "mismatches": [], "error": None}

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

def basic_env():
    e = {"python": sys.version.split()[0], "platform": platform.platform()}
    try:
        import torch
        e.update(torch=torch.__version__, torch_cuda_build=torch.version.cuda, cuda_available=bool(torch.cuda.is_available()),
                 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    except Exception as ex:
        e["torch_error"] = f"{type(ex).__name__}: {ex}"
    try:
        e["nvidia_smi"] = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout[-1500:] or None
    except Exception:
        e["nvidia_smi"] = None
    return e

def mark(step, state, detail=""):
    STATUS["steps"][step] = {"state": state, "detail": detail}
    print(f"[{state}] {step} {detail}")

def build_manifest(aborted):
    steps = STATUS["steps"]
    failed = [k for k, v in steps.items() if v["state"] == "FAILED"]
    blocked = [k for k, v in steps.items() if v["state"] == "BLOCKED"]
    ok = lambda s: steps.get(s, {}).get("state") == "SUCCESS"
    rm, rep = CTX.get("run_manifest"), CTX.get("benchmark")
    kind = CTX.get("video_kind")
    gpu_ok = bool(ok("run_perception") and ok("benchmark") and ok("unit_tests") and rm and rep and not rm.get("is_mock")
                  and rm["components"]["detector"]["parameters"].get("device", "").startswith("cuda")
                  and (CTX.get("environment_full") or {}).get("cuda", {}).get("available"))
    return {
        "manifest_version": "colab-run-manifest-0.2.0-task1",
        "status": "ABORTED" if aborted else ("COMPLETED_WITH_FAILURES" if (failed or blocked) else "COMPLETED"),
        "started_utc": STATUS["started_utc"], "written_utc": NOW().isoformat(),
        "first_failed_stage": next((k for k, v in steps.items() if v["state"] in ("FAILED", "BLOCKED")), None),
        "failed_stages": failed, "blocked_stages": blocked, "errors": ERRORS, "steps": steps,
        "blocked_validations": STATUS["blocked_validations"],
        "environment_basic": CTX.get("environment_basic") or basic_env(), "environment_full": CTX.get("environment_full"),
        "bundle_sha256": CTX.get("zip_sha256"), "expected_zip_sha256_supplied": bool(EXPECTED_ZIP_SHA256.strip()),
        "weights": CTX.get("weights"), "video_kind": kind, "input_video": CTX.get("input_video"),
        "benchmark_marked_plumbing_only": CTX.get("benchmark_plumbing_only"),
        "gpu_execution_succeeded": gpu_ok,
        "real_video_run_completed": bool(gpu_ok and kind == "real_road_video"),
        "real_video_detection_quality_validated": False,
        "performance_claims_supported": bool(rep and rep.get("supports_performance_claims") and kind == "real_road_video"),
        "notes": ["detection QUALITY is never validated by this notebook: no ground truth exists",
                  "a synthetic-video benchmark is --plumbing-only and can never support a performance claim",
                  "gpu_execution_succeeded is true only if the tests, the run and the benchmark all succeeded with a CUDA detector"],
        "local_experiment_dir": str(LOCAL_EXP), "drive_experiment_dir": None if DRIVE_EXP is None else str(DRIVE_EXP), "drive_copy": DRIVE_COPY,
    }

def _write_json(path, obj):
    pathlib.Path(path).write_text(json.dumps(obj, indent=2))

def finalize(aborted=False):
    # Write colab_run_manifest.json locally, copy the experiment folder to Drive, read it back, and record the result. Never raises.
    try:
        m = build_manifest(aborted)
        _write_json(LOCAL_EXP / "colab_run_manifest.json", m)
        if DRIVE_EXP is not None:
            DRIVE_COPY["attempted"] = True
            DRIVE_COPY["mismatches"], DRIVE_COPY["error"] = [], None
            try:
                n = 0
                for src in sorted(p for p in LOCAL_EXP.rglob("*") if p.is_file()):
                    rel = src.relative_to(LOCAL_EXP)
                    dst = DRIVE_EXP / rel
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dst)
                    n += 1
                    if dst.stat().st_size != src.stat().st_size or (src.stat().st_size < 256 * 2**20 and sha256(dst) != sha256(src)):
                        DRIVE_COPY["mismatches"].append(str(rel))
                DRIVE_COPY["files"] = n
                DRIVE_COPY["ok"] = not DRIVE_COPY["mismatches"]
            except Exception as ex:
                DRIVE_COPY["ok"], DRIVE_COPY["error"] = False, f"{type(ex).__name__}: {ex}"
            m = build_manifest(aborted)
            _write_json(LOCAL_EXP / "colab_run_manifest.json", m)
            try:
                _write_json(DRIVE_EXP / "colab_run_manifest.json", m)
            except Exception as ex:
                DRIVE_COPY["ok"], DRIVE_COPY["error"] = False, f"manifest copy failed: {ex}"
        print(f"[manifest] {LOCAL_EXP / 'colab_run_manifest.json'}  status={m['status']}  drive_copy_ok={DRIVE_COPY['ok']}")
        return m
    except Exception as ex:
        print("[manifest] FAILED to write the manifest:", type(ex).__name__, ex)

@contextlib.contextmanager
def stage(name, blocked_on=()):
    # Run one notebook stage. On ANY exception: record the stage, the error and a traceback, write the manifest, copy to Drive, re-raise.
    mark(name, "RUNNING")
    try:
        yield
    except BaseException as exc:
        state = "BLOCKED" if (blocked_on and isinstance(exc, blocked_on)) else "FAILED"
        STATUS["steps"][name] = {"state": state, "detail": f"{type(exc).__name__}: {exc}"}
        ERRORS.append({"stage": name, "state": state, "type": type(exc).__name__, "message": str(exc), "traceback_tail": traceback.format_exc()[-3000:]})
        print(f"[{state}] {name}: {type(exc).__name__}: {exc}")
        finalize(aborted=True)
        raise
    else:
        if STATUS["steps"][name]["state"] == "RUNNING":
            mark(name, "SUCCESS", STATUS["steps"][name]["detail"])

CTX["environment_basic"] = basic_env()
print("local experiment directory:", LOCAL_EXP)
print(json.dumps(CTX["environment_basic"], indent=1)[:1500])

In [ ]:
# ---- 3. GPU check ---------------------------------------------------------------------------------------------------------
with stage("gpu_check"):
    import torch
    print(CTX["environment_basic"].get("nvidia_smi") or "nvidia-smi not available (no NVIDIA GPU runtime?)")
    print("python", sys.version.split()[0], "| torch", torch.__version__, "| torch CUDA build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("no GPU: Runtime -> Change runtime type -> T4 GPU, then re-run (nothing was validated)")
    GPU_NAME = torch.cuda.get_device_name(0)
    CTX["gpu"] = GPU_NAME
    print("GPU:", GPU_NAME)
    if "T4" not in GPU_NAME:
        print("NOTE: this notebook targets a Tesla T4; results on", GPU_NAME, "are recorded as such and are not T4 results.")
    STATUS["steps"]["gpu_check"]["detail"] = GPU_NAME

In [ ]:
# ---- 4. Mount Drive and reserve the UNIQUE Drive experiment folder (results are copied there at the end) -------------------
with stage("drive_mount"):
    from google.colab import drive
    drive.mount("/content/drive")
    exp = pathlib.Path(DRIVE_ROOT) / "experiments" / f"{STAMP}_{EXPERIMENT_NAME}"
    exp.mkdir(parents=True, exist_ok=False)          # never reuses an existing experiment
    DRIVE_EXP = exp
    print("Drive experiment directory:", DRIVE_EXP)

In [ ]:
# ---- 5. Load the uploaded ZIP and VERIFY SHA-256 before anything from it is executed --------------------------------------
with stage("bundle_hash_verification"):
    zip_path, man_path = pathlib.Path(DRIVE_ROOT) / BUNDLE_ZIP, pathlib.Path(DRIVE_ROOT) / BUNDLE_MANIFEST
    if not zip_path.is_file():
        raise FileNotFoundError(f"upload {BUNDLE_ZIP} to {DRIVE_ROOT} first")
    if not man_path.is_file():
        raise FileNotFoundError(f"upload {BUNDLE_MANIFEST} to {DRIVE_ROOT} first")
    manifest = json.loads(man_path.read_text())
    zsha = sha256(zip_path)
    CTX["zip_sha256"] = zsha
    if zsha != manifest["bundle_sha256"]:
        raise ValueError(f"ZIP hash {zsha} does not match the manifest {manifest['bundle_sha256']}")
    if EXPECTED_ZIP_SHA256.strip() and zsha != EXPECTED_ZIP_SHA256.strip().lower():
        raise ValueError(f"ZIP hash {zsha} does not match EXPECTED_ZIP_SHA256 {EXPECTED_ZIP_SHA256.strip().lower()}")
    SRC = pathlib.Path("/content/pedsafety_src")
    if SRC.exists():
        shutil.rmtree(SRC)
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        if names != set(manifest["files"]):
            raise ValueError("ZIP contents differ from the manifest file list")
        for n in names:
            if n.startswith("/") or ".." in n.split("/"):
                raise ValueError(f"unsafe path in ZIP: {n}")
        zf.extractall(SRC)
    bad = [n for n, e in manifest["files"].items() if sha256(SRC / n) != e["sha256"]]
    if bad:
        raise ValueError(f"hash mismatch after extraction: {bad[:5]}")
    shutil.copy(man_path, LOCAL_EXP / BUNDLE_MANIFEST)
    os.chdir(SRC)
    STATUS["steps"]["bundle_hash_verification"]["detail"] = f"{len(names)} files, zip sha256 {zsha}"

In [ ]:
# ---- 6. Install pinned dependencies (Colab keeps its own CUDA torch and OpenCV; they are NOT reinstalled) ------------------
with stage("dependency_install"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.212", "lapx==0.13.1", "psutil", "pytest"],
                       capture_output=True, text=True)
    (LOCAL_EXP / "logs" / "pip_install.log").write_text(r.stdout + "\n" + r.stderr)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if r.returncode != 0:
        raise RuntimeError("pip install failed (exit %d). Last output: %s" % (r.returncode, (r.stderr or r.stdout)[-1200:]))

In [ ]:
# ---- 7. Record Python / PyTorch / CUDA / GPU / library versions -----------------------------------------------------------
with stage("environment_record"):
    sys.path.insert(0, str(SRC))
    from pedsafety.perception.runtime_info import collect_environment, hardware_string
    ENV = collect_environment()
    CTX["environment_full"] = ENV
    _write_json(LOCAL_EXP / "environment.json", ENV)
    print(json.dumps({k: ENV[k] for k in ("python", "platform", "cpu", "cpu_count", "ram_total_gb")}, indent=1))
    print(json.dumps(ENV["libraries"], indent=1)); print(json.dumps(ENV["cuda"], indent=1))
    print("hardware declared:", hardware_string(ENV))

In [ ]:
# ---- 8. Download the approved YOLO weights (Colab only; never committed), check them, and record provenance ----------------
with stage("weights_download", blocked_on=(urllib.error.URLError, TimeoutError, ConnectionError)):
    import urllib.request
    WEIGHTS_DIR = pathlib.Path("/content/weights"); WEIGHTS_DIR.mkdir(exist_ok=True)
    W = WEIGHTS_DIR / WEIGHTS_FILE
    part = WEIGHTS_DIR / (WEIGHTS_FILE + ".part")
    with urllib.request.urlopen(WEIGHTS_URL, timeout=120) as resp, open(part, "wb") as out:
        shutil.copyfileobj(resp, out)
    size = part.stat().st_size
    with open(part, "rb") as fh:
        magic = fh.read(4)
    if size < MIN_WEIGHTS_BYTES:
        raise ValueError(f"downloaded weights are only {size} bytes (< {MIN_WEIGHTS_BYTES}); probably an error page, not a model: {WEIGHTS_URL}")
    if magic[:2] != b"PK":
        raise ValueError(f"downloaded file does not look like a PyTorch checkpoint (first bytes {magic!r}): {WEIGHTS_URL}")
    part.replace(W)
    import ultralytics
    prov = {"file": WEIGHTS_FILE, "source_url": WEIGHTS_URL, "downloaded_utc": NOW().isoformat(), "bytes": size, "sha256": sha256(W),
            "size_check": f">= {MIN_WEIGHTS_BYTES} bytes and zip/PK header: passed", "ultralytics_version": ultralytics.__version__,
            "licence": "AGPL-3.0 (Ultralytics code and pretrained weights) - review https://www.ultralytics.com/license before any redistribution",
            "model_version_note": "version of the weights is the release tag in the URL; sha256 above is the authoritative identity (no reference hash was available)"}
    CTX["weights"] = prov
    _write_json(LOCAL_EXP / "weights_provenance.json", prov)
    print(json.dumps(prov, indent=2))
    STATUS["steps"]["weights_download"]["detail"] = prov["sha256"][:16]

In [ ]:
# ---- 9. Run the Task 1 tests inside Colab (same test files as local) ------------------------------------------------------
with stage("unit_tests"):
    t = subprocess.run([sys.executable, "-W", "ignore", "-m", "pytest", "tests/phase3", "-k", "task1", "-q", "-rs", "-p", "no:cacheprovider"],
                       capture_output=True, text=True, cwd=SRC)
    (LOCAL_EXP / "logs" / "pytest_task1.log").write_text(t.stdout + "\n" + t.stderr)
    print(t.stdout[-3000:])
    lines = [l for l in t.stdout.strip().splitlines() if l.strip()]
    tests = {"returncode": t.returncode, "summary_line": lines[-1] if lines else "(no pytest output)"}
    _write_json(LOCAL_EXP / "tests_summary.json", tests)
    if t.returncode == 0:
        STATUS["steps"]["unit_tests"]["detail"] = tests["summary_line"]
    else:
        mark("unit_tests", "FAILED", tests["summary_line"])          # recorded; the run continues so the log is complete
        ERRORS.append({"stage": "unit_tests", "state": "FAILED", "type": "PytestFailure", "message": tests["summary_line"], "traceback_tail": t.stdout[-2000:]})

In [ ]:
# ---- 10. Configure the input video (a real video is copied OFF Drive to local storage first) ------------------------------
with stage("input_video"):
    inp = pathlib.Path(WORK_ROOT) / "input"; inp.mkdir(parents=True, exist_ok=True)
    if INPUT_VIDEO:
        srcv = pathlib.Path(INPUT_VIDEO)
        if not srcv.is_file():
            raise FileNotFoundError(f"INPUT_VIDEO does not exist: {INPUT_VIDEO}")
        VIDEO = str(inp / srcv.name)
        shutil.copy2(srcv, VIDEO)
        if sha256(VIDEO) != sha256(srcv):
            raise ValueError("local copy of the input video differs from the Drive original")
        VIDEO_KIND = "real_road_video"
        CTX["input_video"] = {"name": srcv.name, "bytes": os.path.getsize(VIDEO), "sha256": sha256(VIDEO), "copied_to": VIDEO}
        STATUS["steps"]["input_video"]["detail"] = f"real video {srcv.name} copied to local storage, sha256 {CTX['input_video']['sha256'][:16]}"
    else:
        VIDEO = str(LOCAL_EXP / "synthetic_scene.mp4")
        r = subprocess.run([sys.executable, "scripts/make_synthetic_video.py", "--out", VIDEO], capture_output=True, text=True, cwd=SRC)
        print(r.stdout, r.stderr)
        if r.returncode != 0:
            raise RuntimeError("synthetic video generation failed: " + (r.stderr or r.stdout)[-800:])
        VIDEO_KIND = "synthetic_plumbing_only"
        CTX["input_video"] = {"name": "synthetic_scene.mp4", "synthetic": True}
        STATUS["steps"]["input_video"]["detail"] = "synthetic rectangles (plumbing/timing only)"
        STATUS["blocked_validations"]["real_video_validation"] = "no INPUT_VIDEO supplied: the pipeline was NOT run on a real road video"
        print("[BLOCKED VALIDATION] real_video_validation: no INPUT_VIDEO supplied")
    CTX["video_kind"] = VIDEO_KIND
    print("video:", VIDEO, "| kind:", VIDEO_KIND)

In [ ]:
# ---- 11. Run the SAME pipeline script used locally (YOLO + ByteTrack on the GPU); output stays on local storage ------------
import glob
with stage("run_perception"):
    cmd = [sys.executable, "-W", "ignore", "scripts/run_perception.py", "--video", VIDEO, "--model", str(W), "--device", DEVICE, "--save-video",
           "--output-root", str(LOCAL_EXP / "run"), "--run-name", "perception"]
    if MAX_FRAMES: cmd += ["--max-frames", str(MAX_FRAMES)]
    p = subprocess.run(cmd, capture_output=True, text=True, cwd=SRC)
    (LOCAL_EXP / "logs" / "run_perception.log").write_text(p.stdout + "\n" + p.stderr)
    print(p.stdout[-2500:], p.stderr[-1500:])
    runm = sorted(glob.glob(str(LOCAL_EXP / "run" / "*" / "run_manifest.json")))
    if runm:
        CTX["run_manifest"] = json.loads(pathlib.Path(runm[-1]).read_text())
    if p.returncode != 0:
        mark("run_perception", "FAILED", f"exit {p.returncode}")
        ERRORS.append({"stage": "run_perception", "state": "FAILED", "type": "NonZeroExit", "message": f"exit {p.returncode}", "traceback_tail": (p.stderr or p.stdout)[-2000:]})
    else:
        STATUS["steps"]["run_perception"]["detail"] = f"exit 0, outputs: {CTX['run_manifest']['outputs'] if runm else 'manifest missing'}"

In [ ]:
# ---- 12. Benchmark (decode-only / detection-only / tracking-only / combined). Synthetic video => --plumbing-only -----------
with stage("benchmark"):
    plumbing = (VIDEO_KIND != "real_road_video")
    cmd = [sys.executable, "-W", "ignore", "scripts/benchmark_perception.py", "--video", VIDEO, "--model", str(W), "--device", DEVICE, "--warmup", str(WARMUP),
           "--output-root", str(LOCAL_EXP / "benchmark"), "--run-name", "perception"]
    if MAX_FRAMES: cmd += ["--max-frames", str(MAX_FRAMES)]
    if plumbing: cmd += ["--plumbing-only"]
    CTX["benchmark_plumbing_only"] = plumbing
    b = subprocess.run(cmd, capture_output=True, text=True, cwd=SRC)
    (LOCAL_EXP / "logs" / "benchmark.log").write_text(b.stdout + "\n" + b.stderr)
    print(b.stdout[-3000:], b.stderr[-1500:])
    bench = sorted(glob.glob(str(LOCAL_EXP / "benchmark" / "*" / "benchmark.json")))
    if bench:
        CTX["benchmark"] = json.loads(pathlib.Path(bench[-1]).read_text())
    if b.returncode != 0:
        mark("benchmark", "FAILED", f"exit {b.returncode}")
        ERRORS.append({"stage": "benchmark", "state": "FAILED", "type": "NonZeroExit", "message": f"exit {b.returncode}", "traceback_tail": (b.stderr or b.stdout)[-2000:]})
    elif plumbing and CTX.get("benchmark", {}).get("supports_performance_claims") is not False:
        raise RuntimeError("guard: a synthetic-video benchmark must never support performance claims, but benchmark.json says otherwise")
    else:
        STATUS["steps"]["benchmark"]["detail"] = "plumbing-only (synthetic video): not a performance result" if plumbing else "real video"

In [ ]:
# ---- 13. Final manifest + copy of the whole experiment folder to Drive (read back and checked) -----------------------------
final = finalize(aborted=False)
print(json.dumps({k: final[k] for k in ("status", "first_failed_stage", "failed_stages", "blocked_stages", "gpu_execution_succeeded", "real_video_run_completed",
                                         "performance_claims_supported", "benchmark_marked_plumbing_only", "blocked_validations")}, indent=2))
print("drive copy:", DRIVE_COPY)
print("local :", LOCAL_EXP)
print("Drive :", DRIVE_EXP)